# Day 4: HuggingFace Transformers - Solutions

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/09_HuggingFace_Transformers_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Important:** Try the exercises yourself before looking at these solutions!

These are reference solutions for the three exercises in `09_HuggingFace_Transformers.ipynb`. Your own answers may look different and still be correct.

---

In [ ]:
# === COURSE SETUP — run this cell first! ===
# Installs the required packages (most are preinstalled on Google Colab).
%pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow transformers datasets

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from transformers import pipeline
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Setup complete — you are ready to go!")

## Exercise 1: Customer Review Analyzer

Analyze a batch of product reviews, print a formatted table, compute summary stats, and visualize.

In [ ]:
# Load the sentiment pipeline (small, CPU-friendly DistilBERT model ~260 MB)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    framework="tf",
)

reviews = [
    "The product arrived quickly and works perfectly. Very happy with my purchase!",
    "Terrible quality. Broke after just two days of use.",
    "Good value for the price. Not amazing but does the job.",
    "Absolutely love it! Best purchase I've made this year.",
    "The color is different from what was shown. Disappointed.",
    "Shipping was slow but the product itself is great.",
    "Would not recommend. Wasted my money.",
    "Exceeded my expectations! Will buy again.",
]

# Analyze all reviews in one call
review_results = sentiment_analyzer(reviews)

# Formatted table
header = "{:<10} {:>10}   {}".format("Sentiment", "Confidence", "Review")
print(header)
print("=" * 80)
for review, res in zip(reviews, review_results):
    preview = review[:45] + ("..." if len(review) > 45 else "")
    print("{:<10} {:>9.1%}   {}".format(res["label"], res["score"], preview))

# Summary stats
n_pos = sum(1 for r in review_results if r["label"] == "POSITIVE")
n_neg = len(review_results) - n_pos
avg_conf = np.mean([r["score"] for r in review_results])
print("\nSummary:")
print("  Positive: {}   Negative: {}".format(n_pos, n_neg))
print("  Average confidence: {:.1%}".format(avg_conf))

In [ ]:
# Visualize: one bar per review, signed score (positive=green, negative=red)
scores = [r["score"] if r["label"] == "POSITIVE" else -r["score"] for r in review_results]
colors = ["green" if s > 0 else "red" for s in scores]

plt.figure(figsize=(12, 6))
plt.bar(range(len(scores)), scores, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(range(len(reviews)), ["#{}".format(i + 1) for i in range(len(reviews))])
plt.ylabel("Sentiment score  (+ positive / - negative)")
plt.title("Customer Review Sentiment")
plt.ylim(-1.1, 1.1)
plt.show()

### Explanation:

- We pass the **whole list** of reviews to the pipeline at once — it batches them efficiently and returns one `{label, score}` dict per review.
- For the chart we turn each result into a **signed score**: positive reviews get `+score`, negative get `-score`. One bar chart then shows both direction (color and sign) and confidence (bar height).
- For larger datasets you'd typically load the results into a DataFrame and use `value_counts()`.

## Exercise 2: News Article Classifier

Use **zero-shot classification** to sort headlines into your own categories — no training required.

In [ ]:
# Load the zero-shot pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    framework="tf",
)

headlines = [
    "Central bank raises interest rates to combat inflation",
    "Local team wins championship in dramatic overtime final",
    "New smartphone features foldable screen and AI assistant",
    "Study links daily exercise to lower risk of heart disease",
    "Government announces new climate policy ahead of summit",
]

categories = ["politics", "sports", "technology", "health", "business"]

print("{:<14} {:>6}   {}".format("Top category", "Conf.", "Headline"))
print("=" * 80)
for headline in headlines:
    result = classifier(headline, categories)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print("{:<14} {:>5.0%}   {}".format(top_label, top_score, headline[:50]))

### Explanation:

- **Zero-shot** means the model was never trained on *these specific* categories — we just hand it the label words and it scores how well each fits the text.
- `result["labels"]` and `result["scores"]` come back **sorted best-first**, so index `[0]` is the top prediction.
- Try adding a category (e.g. `"entertainment"`) and rerun — no retraining needed. That flexibility is the whole point of zero-shot classification.

## Exercise 3: Explore a HuggingFace Dataset

Load a dataset, inspect its structure, convert to Pandas, and apply a pipeline to a few examples. Here we use `rotten_tomatoes` (a small movie-review sentiment dataset).

In [ ]:
# Load a dataset from the HuggingFace Hub
my_dataset = load_dataset("rotten_tomatoes")

# Explore the structure
print(my_dataset)
print("\nTrain examples: {:,}".format(len(my_dataset["train"])))
print("Features:", my_dataset["train"].features)
print("\nFirst example:", my_dataset["train"][0])

In [ ]:
# Convert the training split to Pandas and show basic statistics
df = my_dataset["train"].to_pandas()
print(df.head())

print("\nLabel distribution:")
print(df["label"].value_counts().rename({0: "negative", 1: "positive"}))

df["word_count"] = df["text"].str.split().str.len()
print("\nAverage review length: {:.0f} words".format(df["word_count"].mean()))

In [ ]:
# Apply the sentiment pipeline to a few examples and compare to the true labels
samples = my_dataset["test"][:5]
texts = [t[:512] for t in samples["text"]]   # truncate to model max length
true_labels = samples["label"]

preds = sentiment_analyzer(texts)

print("{:<10} {:<10} {:>6}   {}".format("True", "Predicted", "Conf.", "Review preview"))
print("=" * 80)
for true_label, pred, text in zip(true_labels, preds, texts):
    true_str = "positive" if true_label == 1 else "negative"
    pred_str = pred["label"].lower()
    mark = "✓" if true_str == pred_str else "✗"
    print("{:<10} {:<10} {:>5.0%}  {} {}...".format(true_str, pred_str, pred["score"], mark, text[:38]))

### Explanation:

- `load_dataset("rotten_tomatoes")` downloads and caches the dataset in one line and returns standard `train`/`validation`/`test` splits.
- `.to_pandas()` turns a split into a familiar DataFrame so we can reuse everything from the earlier notebooks (`value_counts`, string operations, etc.).
- We **truncate** each review to 512 characters because the model has a maximum input length.
- Reusing the same `sentiment_analyzer` on a brand-new dataset shows the strength of pre-trained models: one model, many datasets, no retraining.

**Other datasets to try:** `"ag_news"` (news topics — pair with zero-shot), `"emotion"` (emotion labels), `"imdb"` (longer reviews).

---

## Done!

You've solved all three exercises using only pre-trained models and pipelines. Notice how little code each task needed — that's the power of transfer learning with HuggingFace.